# Processamento do Dataset visual-memory/ConvAI2

Este notebook processa todos os splits de `visual-memory/ConvAI2` para:
1. Validar e canonicalizar as personas revisadas, ignorando a ordem das frases
2. Gerar IDs SHA-256 determinísticos a partir das personas canonicalizadas
3. Criar um **dataset enriquecido** com `your_persona_id` e `partner_persona_id`
4. Criar um **dataset de mapeamento** com cada ID e sua persona revisada
5. Validar a integridade dos datasets antes da publicação

> As listas originais de persona são preservadas no dataset enriquecido. A ordenação é usada apenas para definir a identidade da persona e o texto do mapping.

In [1]:
import hashlib
import re
from typing import Any

from datasets import Dataset, DatasetDict, load_dataset

SOURCE_REPO_ID = "visual-memory/ConvAI2"
ENRICHED_REPO_ID = "visual-memory/ConvAI2-With-Ids"
MAPPING_REPO_ID = "visual-memory/ConvAI2-Mapping"

PERSONA_FIELDS = (
    {"role": "your", "persona": "your_persona", "id": "your_persona_id"},
    {
        "role": "partner",
        "persona": "partner_persona",
        "id": "partner_persona_id",
    },
)
REQUIRED_COLUMNS = {fields["persona"] for fields in PERSONA_FIELDS}
ID_COLUMNS = [fields["id"] for fields in PERSONA_FIELDS]
SHA256_PATTERN = re.compile(r"^[0-9a-f]{64}$")


def canonicalize_persona(
    value: Any, *, column: str, row_context: str
) -> tuple[str, ...]:
    """Valida uma persona e retorna suas frases limpas em ordem canônica."""
    if not isinstance(value, list):
        raise ValueError(
            f"Valor invalido em {column} ({row_context}): esperado list[str], "
            f"recebido {type(value).__name__}."
        )
    if not value:
        raise ValueError(f"Persona vazia em {column} ({row_context}).")

    invalid_items = [
        index for index, phrase in enumerate(value) if not isinstance(phrase, str)
    ]
    if invalid_items:
        raise ValueError(
            f"Itens invalidos em {column} ({row_context}): os indices "
            f"{invalid_items} nao contem strings."
        )

    stripped_phrases = [phrase.strip() for phrase in value]
    empty_items = [
        index for index, phrase in enumerate(stripped_phrases) if not phrase
    ]
    if empty_items:
        raise ValueError(
            f"Frases vazias em {column} ({row_context}): indices {empty_items}."
        )

    return tuple(sorted(stripped_phrases))


def persona_to_text(persona: tuple[str, ...]) -> str:
    """Converte uma persona já canonicalizada em seu texto de mapping."""
    return " ".join(persona)


def generate_persona_id(persona_revised: str) -> str:
    """Gera um ID SHA-256 determinístico para o texto canonicalizado."""
    return hashlib.sha256(persona_revised.encode("utf-8")).hexdigest()

## 1. Carregar e validar o dataset (todos os splits)

In [2]:
dataset_dict = load_dataset(SOURCE_REPO_ID)

missing_columns = {
    split_name: sorted(REQUIRED_COLUMNS - set(dataset.column_names))
    for split_name, dataset in dataset_dict.items()
}
missing_columns = {
    split_name: columns
    for split_name, columns in missing_columns.items()
    if columns
}
if missing_columns:
    raise ValueError(f"Colunas de persona ausentes por split: {missing_columns}")

print(f"Total de linhas (todos os splits): {sum(len(ds) for ds in dataset_dict.values()):,}")
for split_name, dataset in dataset_dict.items():
    print(f"  {split_name}: {len(dataset):,} linhas")

Total de linhas (todos os splits): 18,878
  train: 17,878 linhas
  validation: 1,000 linhas


## 2. Canonicalizar e unificar os registros de persona

A identidade desconsidera a ordem das frases, mas preserva caixa, pontuação e o conteúdo textual. Espaços externos são removidos.

In [3]:
canonical_personas: set[tuple[str, ...]] = set()

for split_name, dataset in dataset_dict.items():
    for row_index, row in enumerate(dataset):
        row_context = f"split={split_name!r}, row={row_index}"
        for fields in PERSONA_FIELDS:
            canonical_personas.add(
                canonicalize_persona(
                    row[fields["persona"]],
                    column=fields["persona"],
                    row_context=row_context,
                )
            )

sorted_canonical_personas = sorted(canonical_personas)
persona_texts = [persona_to_text(persona) for persona in sorted_canonical_personas]

if len(set(persona_texts)) != len(persona_texts):
    raise ValueError(
        "Personas canonicalizadas diferentes produziram o mesmo texto de mapping."
    )

print(f"Personas revisadas unicas: {len(sorted_canonical_personas):,}")
print(f"Exemplo de persona canonicalizada: {persona_texts[0]}")

Personas revisadas unicas: 10,321
Exemplo de persona canonicalizada: a few months ago , i purchased an rv. i am a professional freight driver. i have a gym membership. i like watching the nfl. i used to be enlisted in the army.


## 3. Gerar IDs determinísticos (SHA-256)

In [4]:
canonical_to_id = {
    persona: generate_persona_id(persona_to_text(persona))
    for persona in sorted_canonical_personas
}

if len(set(canonical_to_id.values())) != len(canonical_to_id):
    raise ValueError("Colisao de IDs SHA-256 detectada entre personas diferentes.")

print(f"IDs gerados: {len(canonical_to_id):,}")
print(f"Exemplo de ID: {next(iter(canonical_to_id.values()))}")

IDs gerados: 10,321
Exemplo de ID: b59fdb1948c68774f57f26b1594f41d8c5d4ba9ea64e7498126a043738271a15


## 4. Criar o dataset enriquecido

Os registros originais são copiados sem modificar as listas de persona. Somente as duas colunas de ID são acrescentadas.

In [5]:
enriched_dataset_dict = DatasetDict()

for split_name, dataset in dataset_dict.items():
    enriched_rows = []
    for row_index, row in enumerate(dataset):
        row_context = f"split={split_name!r}, row={row_index}"
        enriched_row = dict(row)
        for fields in PERSONA_FIELDS:
            canonical_persona = canonicalize_persona(
                row[fields["persona"]],
                column=fields["persona"],
                row_context=row_context,
            )
            enriched_row[fields["id"]] = canonical_to_id[canonical_persona]
        enriched_rows.append(enriched_row)

    enriched_dataset_dict[split_name] = Dataset.from_list(enriched_rows)

print("Dataset enriquecido (estrutura DatasetDict):")
for split_name, dataset in enriched_dataset_dict.items():
    print(f"  {split_name}: {len(dataset):,} linhas")
first_split = next(iter(enriched_dataset_dict))
print(f"Colunas: {enriched_dataset_dict[first_split].column_names}")

Dataset enriquecido (estrutura DatasetDict):
  train: 17,878 linhas
  validation: 1,000 linhas
Colunas: ['conversation_id', 'your_persona', 'partner_persona', 'turns', 'source_split', 'your_persona_id', 'partner_persona_id']


## 5. Criar o dataset de mapeamento

In [6]:
mapping_data = [
    {
        "persona-id": canonical_to_id[persona],
        "persona_revised": persona_to_text(persona),
    }
    for persona in sorted_canonical_personas
]
mapping_dataset = Dataset.from_list(mapping_data)

print(f"Dataset de mapeamento: {len(mapping_dataset):,} personas unicas")
print(f"Colunas: {mapping_dataset.column_names}")

Dataset de mapeamento: 10,321 personas unicas
Colunas: ['persona-id', 'persona_revised']


## 6. Validar a integridade dos datasets

In [7]:
assert list(enriched_dataset_dict) == list(dataset_dict), "Os splits foram alterados."
for split_name, original_dataset in dataset_dict.items():
    enriched_dataset = enriched_dataset_dict[split_name]
    assert len(enriched_dataset) == len(original_dataset), (
        f"Quantidade de linhas alterada no split {split_name!r}."
    )
    assert enriched_dataset.column_names == [
        *original_dataset.column_names,
        *ID_COLUMNS,
    ], f"Colunas inesperadas no split {split_name!r}."

    for column in original_dataset.column_names:
        assert enriched_dataset[column] == original_dataset[column], (
            f"Valores originais alterados no split {split_name!r}, "
            f"coluna {column!r}."
        )

assert mapping_dataset.column_names == ["persona-id", "persona_revised"]

mapping_by_id = {}
for mapping_row in mapping_dataset:
    persona_id = mapping_row["persona-id"]
    assert SHA256_PATTERN.fullmatch(persona_id), f"ID invalido: {persona_id!r}"
    assert persona_id not in mapping_by_id, f"ID duplicado: {persona_id!r}"
    assert persona_id == generate_persona_id(mapping_row["persona_revised"]), (
        f"ID inconsistente com persona_revised: {persona_id!r}"
    )
    mapping_by_id[persona_id] = mapping_row

for split_name, dataset in enriched_dataset_dict.items():
    for row_index, row in enumerate(dataset):
        row_context = f"split={split_name!r}, row={row_index}"
        for fields in PERSONA_FIELDS:
            persona_id = row[fields["id"]]
            assert SHA256_PATTERN.fullmatch(persona_id), (
                f"ID invalido em {row_context}, coluna {fields['id']!r}."
            )
            assert persona_id in mapping_by_id, (
                f"ID sem mapping em {row_context}: {persona_id!r}"
            )
            canonical_persona = canonicalize_persona(
                row[fields["persona"]],
                column=fields["persona"],
                row_context=row_context,
            )
            assert persona_id == canonical_to_id[canonical_persona]

assert len(mapping_by_id) == len(canonical_personas)

order_test_context = "validacao de independencia da ordem"
order_test_persona = ["segunda frase", " primeira frase "]
reversed_order_test_persona = list(reversed(order_test_persona))
canonical_a = canonicalize_persona(
    order_test_persona, column="persona", row_context=order_test_context
)
canonical_b = canonicalize_persona(
    reversed_order_test_persona, column="persona", row_context=order_test_context
)
assert canonical_a == canonical_b
assert generate_persona_id(persona_to_text(canonical_a)) == generate_persona_id(
    persona_to_text(canonical_b)
)

print("Todas as validacoes foram concluidas com sucesso.")
sample = enriched_dataset_dict[first_split][0]
print(f"\nExemplo de linha enriquecida (split {first_split!r}):")
print(f"  your_persona_id: {sample['your_persona_id']}")
print(f"  partner_persona_id: {sample['partner_persona_id']}")
print(f"  your_persona preservada: {sample['your_persona']}")
print(f"  partner_persona preservada: {sample['partner_persona']}")

Todas as validacoes foram concluidas com sucesso.

Exemplo de linha enriquecida (split 'train'):
  your_persona_id: af6526011f30a86d5b8733a3d9b48787a5e03b0446163e7774500cede5a3f6a7
  partner_persona_id: 988c793ba87dd93a66f6522676287b4aab61f388ef934bf3546e83704963508f
  your_persona preservada: ['i love to redesign houses.', 'killing for sport is my hobby.', 'i shot an arrow the other day !.', 'i like to get dressed up.']
  partner_persona preservada: ['my favorite hobbies are based on old fashioned life skills.', 'i race large felines who are in captivity to remain healthy.', 'i was a really good runner when i was younger.', 'i am a carnivore.']


## 7. Publicar no Hugging Face Hub

> Execute as duas células abaixo manualmente, somente depois de revisar as validações e as amostras.

In [8]:
from huggingface_hub import notebook_login

notebook_login()

In [9]:
enriched_dataset_dict.push_to_hub(ENRICHED_REPO_ID)
mapping_dataset.push_to_hub(MAPPING_REPO_ID)

print(f"Dataset enriquecido publicado em {ENRICHED_REPO_ID}!")
print(f"Dataset de mapeamento publicado em {MAPPING_REPO_ID}!")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Dataset enriquecido publicado em visual-memory/ConvAI2-With-Ids!
Dataset de mapeamento publicado em visual-memory/ConvAI2-Mapping!
